In [ ]:
!pip install transformers datasets librosa torchaudio accelerate matplotlib captum

In [ ]:
import torch
import numpy as np

from transformers import (
    AutoModelForAudioClassification,
    AutoFeatureExtractor,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

from datasets import load_dataset, Audio, DatasetDict
from sklearn.model_selection import GroupShuffleSplit

###Φόρτωση Συνόλου

In [ ]:
dataset = load_dataset("AbstractTTS/CREMA-D", split="train")
dataset = dataset.cast_column("audio", Audio(sampling_rate=16_000))

print(dataset.features)
print(dataset[0]["file"], dataset[0]["major_emotion"])

In [ ]:
labels = sorted(set(dataset["major_emotion"]))
num_labels = len(labels)
label2id = {name: i for i, name in enumerate(labels)}
id2label = {i: name for i, name in enumerate(labels)}

In [ ]:
def add_speaker(batch):
    batch["actor_id"] = batch["file"].split("_")[0]
    return batch

dataset = dataset.map(add_speaker)
print(len(set(dataset["actor_id"])), "μοναδικοί ομιλητές")

from sklearn.model_selection import GroupShuffleSplit

actor_ids = dataset["actor_id"]
splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(splitter.split(X=range(len(dataset)), groups=actor_ids))

from datasets import DatasetDict
dataset = DatasetDict({
    "train": dataset.select(train_idx),
    "test": dataset.select(test_idx)
})

###Φόρτωση Μοντέλου

In [ ]:
repo = "microsoft/wavlm-base"

feature_extractor = AutoFeatureExtractor.from_pretrained(repo)

model = AutoModelForAudioClassification.from_pretrained(
    repo,
    num_labels=num_labels,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,
).to("cuda")

In [ ]:
MAX_LENGTH = 6 * 16000

def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_values"] = feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
        max_length=MAX_LENGTH,
        truncation=True
    ).input_values[0]
    batch["labels"] = label2id[batch["major_emotion"]]
    return batch

encoded_dataset = dataset.map(prepare_dataset, remove_columns=dataset["train"].column_names)

In [ ]:
# Data Collator
def collate_fn(features):
    input_features = [{"input_values": f["input_values"]} for f in features]
    batch = feature_extractor.pad(input_features, padding=True, return_tensors="pt")
    batch["labels"] = torch.tensor([f["labels"] for f in features], dtype=torch.long)
    return batch

data_collator = collate_fn

In [ ]:
# Μετρικές εκπαίδευσης
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, matthews_corrcoef, cohen_kappa_score,
    roc_auc_score, log_loss, confusion_matrix, average_precision_score
)
from sklearn.preprocessing import label_binarize
from scipy.special import softmax

all_confusion_matrices = []

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    probs = softmax(logits, axis=1)

    num_classes = probs.shape[1]
    labels_bin = label_binarize(labels, classes=list(range(num_classes)))

    roc_auc = roc_auc_score(labels_bin, probs, multi_class="ovr", average="macro")
    pr_auc = average_precision_score(labels_bin, probs, average="macro")

    accuracy = accuracy_score(labels, predictions)
    balanced_accuracy = balanced_accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, average="weighted", zero_division=0)
    recall = recall_score(labels, predictions, average="weighted")
    f1 = f1_score(labels, predictions, average="weighted")
    mcc = matthews_corrcoef(labels, predictions)
    k_score = cohen_kappa_score(labels, predictions)
    logloss = log_loss(labels, probs)

    all_confusion_matrices.append(confusion_matrix(labels, predictions))

    return {
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mcc": mcc,
        "k_score": k_score,
        "log_loss": logloss,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
    }

In [ ]:
# Υπερπαράμετροι εκπαίδευσης
per_device_train_batch_size = 8
gradient_accumulation_steps = 4
num_train_epochs = 8

effective_batch_size = per_device_train_batch_size * gradient_accumulation_steps
steps_per_epoch = len(encoded_dataset["train"]) // effective_batch_size
total_steps = steps_per_epoch * num_train_epochs
warmup_steps = max(1, int(0.1 * total_steps))

training_args = TrainingArguments(
    output_dir="./wavlm-finetuned-cremad",
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=gradient_accumulation_steps,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    num_train_epochs=num_train_epochs,
    fp16=True,
    learning_rate=3e-5,
    warmup_steps=warmup_steps,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
)

In [ ]:
all_confusion_matrices.clear()

early_stopping = EarlyStoppingCallback(early_stopping_patience=2)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"],
    data_collator=data_collator,
    processing_class=feature_extractor,
    compute_metrics=compute_metrics,
    callbacks=[early_stopping],
)

print("Εκκίνηση εκπαίδευσης...")
trainer.train()
print("Εκπαίδευση ολοκληρώθηκε!")

In [ ]:
# Τελικό evaluation του best model
eval_results = trainer.evaluate()
print(eval_results)

In [ ]:
#Αποθήκευση των ετικετών για τους άξονες των matrices
labels = [
    model.config.id2label[i]
    for i in sorted(model.config.id2label.keys())
]

#Loop στην λίστα με τα confusion matrices
for epoch, cm in enumerate(all_confusion_matrices):
  #Δημιοργία γραφήματος
  plt.figure(figsize=(6,5))

  #Δηνιουργία Heatmap
  sns.heatmap(
      cm,
      annot=True,
      fmt="d",
      xticklabels=labels,
      yticklabels=labels
  )

  plt.title(f"Confusion Matrix - Epoch {epoch+1}")
  plt.xlabel("Predicted")
  plt.ylabel("True")
  plt.show()

In [ ]:
# Push στο HuggingFace Hub
from huggingface_hub import login

login()

repo_id = "VasilisAsim/wavlm-finetuned-cremad"

model.push_to_hub(repo_id)
feature_extractor.push_to_hub(repo_id)